In [1]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
import time

# --- 🚀 MAXIMUM TPU LOAD SETTINGS (OPTIMIZED) 🚀 ---
MAX_RECURSION_DEPTH = 1_000_000  # 🔥 Highest recursion depth
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 50_000_000  # 🔥 Large-scale TPU batch processing

# ✅ **Vectorized Dynamic Functions**
@jit
def dynamic_pi(depth, scale_factor):
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """🔥 Normalize depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth, scale_factor=1.0):
    """🔥 TPU-Optimized Recursive Computation"""
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        return jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))

    return lax.fori_loop(0, depth, body_fn, x)

# --- ✅ 🚀 TPU SHARDING & PREFETCH ---
devices = jax.devices()
sharding = jax.sharding.PositionalSharding(devices)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=MAX_RECURSION_DEPTH, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ **Progressive Depth Scaling Execution**
def process_with_scaling(x, total_depth):
    """🔥 Gradually increases depth step size to prevent 250K bottleneck."""
    depth_steps = [50_000, 100_000, 200_000, 400_000, 800_000, 1_000_000]
    for step in depth_steps:
        if step > total_depth:
            break
        x = dppu_with_dynamic_pi_phi(x, depth=step)
    return x

# --- ✅ 🚀 EXECUTE AT MAXIMUM TPU LOAD (REVERSE SCALING) ---
for depth in [250_000, 500_000, 1_000_000]:  # 🔥 Smooth depth scaling
    start_time = time.time()
    output_batch = process_with_scaling(batch_input, depth)
    end_time = time.time()
    print(f"✅ Batch Output Shape (Depth={depth}):", output_batch.shape)
    print(f"🔥 Execution Time: {end_time - start_time:.6f} sec")

NUM_TRIALS = 2  # 🔥 Reduce trials to avoid unnecessary TPU overload

# ✅ **Benchmark Execution with Progressive Scaling**
for depth in [250_000, 500_000, 1_000_000]:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_scaling(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)  # ✅ Fetch results asynchronously
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

# ✅ **Investigate TPU Compilation Efficiency**
compiled_fn_250k = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=250_000)
compiled_fn_1M = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=1_000_000)

print("\n🚀 XLA Compilation for Depth=250,000:")
print(compiled_fn_250k.as_text())

print("\n🚀 XLA Compilation for Depth=1,000,000:")
print(compiled_fn_1M.as_text())




NameError: name 'partial' is not defined